# Assignment 15 — Creative Generative AI

Converted from the supplied PDF. Explanations are Markdown cells; Python programs are code cells.

Assignment 15

Creative Generative AI | Ethics | Hyper-Personalization | Reinforcement Learning for Generative Models |
Multimodal Systems

Question 1

Explain how generative AI is used in creative domains: poetry, storytelling, music, art and video. For each domain
name one kind of model or tool, what input it takes and what it produces. Explain the difference between AI as a tool
that assists a creator and AI that replaces a creator.

Answer:

Generative AI models learn the patterns of a very large collection of existing creative work and then produce
new content that follows those patterns. The same idea (learn the distribution, then sample from it) is used in
every creative domain; only the data type and the model family change.

Domain
Example model / tool
Input it takes
What it produces

Poetry
Large language model (LLM) such
as GPT or Claude

Text prompt: theme, form (haiku,
sonnet), tone, rhyme rules

A poem as text

Storytelling
LLM with a long context window
(ChatGPT, Claude)

Premise, characters, genre, length, point
of view

Story, outline, chapter or
dialogue as text

Music
MusicGen (Meta), Suno, Udio,
MuseNet

Text description (for example 'calm lo-fi
piano'), optionally a melody or lyrics

Audio clip or song (or MIDI
notes)

Art
Stable Diffusion, DALL-E,
Midjourney (diffusion models)

Text prompt, optional reference image or
sketch

A still image

Video
Sora, Runway Gen models, Veo
(video diffusion / transformers)

Text prompt, a starting image or a short
clip

A short video clip (frames plus
motion)

How each one works in one line

• Poetry and storytelling: the LLM predicts the next word (token) repeatedly, so the text grows one token at
a time.

• Music: a transformer predicts the next audio or note token, or a diffusion model denoises a spectrogram
into sound.

• Art: a diffusion model starts from random noise and removes noise step by step, guided by the text
embedding.

• Video: the same denoising idea is applied to a stack of frames so objects stay consistent over time.

AI as a tool that assists vs AI that replaces a creator

Aspect
AI assists the creator
AI replaces the creator

Who decides?
The human sets the idea, taste and final choices; AI
offers options.

The AI (or the company running it) makes the creative
decisions end to end.

Role of AI
Brainstorming, drafts, variations, cleaning up,
translation, accessibility.

Produces the finished work with little or no human
input.

Example
A songwriter asks AI for 10 chord ideas, picks one
and rewrites it. A designer uses AI to fill a background
and then paints over it.

A studio generates a whole album or all product
illustrations by one prompt and ships it, with no artist
involved.

Effect
Faster work, more experiments, the creator's voice
stays visible.

Lower cost but fewer jobs, generic style, unclear
credit and ownership.

Conclusion: assistance keeps a human in control and accountable (the 'human in the loop'); replacement
removes the human. The ethical and legal questions (credit, copyright, jobs) become much sharper in the
replacement case, so most responsible uses treat AI as a collaborator, not an author.

Question 2

Explain how a language model generates a poem or a story: next-token prediction, temperature, top-p and the role
of the prompt. Write three different prompts for a short story about the same idea (a neutral prompt, a style-guided
prompt and a constraint-based prompt) and explain how the outputs would differ. Why do high temperature settings
make writing more creative but less consistent?

Answer:

Next-token prediction

The text is split into tokens (pieces of words). Given all tokens so far, the model outputs a probability for every
token in its vocabulary. One token is chosen, appended to the text, and the process repeats. A whole poem or
story is simply this loop run hundreds of times. For the prefix 'The lighthouse stood on the edge of the ...' the
model may give: sea 64%, cliff 24%, world 9%, storm 3%.

Temperature

Before the probabilities are computed, the model's raw scores (logits z) are divided by the temperature T:

p_i = exp(z_i / T) / sum_j exp(z_j / T)

Token
T = 0.5 (focused)
T = 1.0 (normal)
T = 2.0 (flat)

sea
86.5%
64.4%
45.5%

cliff
11.7%
23.7%
27.6%

world
1.6%
8.7%
16.7%

storm
0.2%
3.2%
10.2%

Low T makes the top token dominate (safe, predictable, repetitive). High T flattens the distribution so unlikely
tokens are chosen more often.

Top-p (nucleus sampling)

Top-p keeps only the smallest group of most likely tokens whose probabilities add up to at least p, throws the
rest away, renormalises, and samples from that group. With top-p = 0.9 and the T = 1.0 column above, the kept
tokens are sea, cliff, world (cumulative 96.8%) and storm is removed. It adapts to the situation: when the
model is confident only a few tokens survive; when it is unsure many survive. It stops very unlikely (nonsense)
tokens while still allowing variety.

Role of the prompt

The prompt is the starting context. It changes the probability of every next token, so it controls topic, genre,
voice, length, format and constraints. Examples, a persona, or rules inside the prompt all steer the output
without changing the model.

Three prompts for the same idea

Idea: a lighthouse keeper finds a message in a bottle.

Prompt type
Prompt
How the output differs

1. Neutral
Write a short story about a lighthouse keeper who
finds a message in a bottle.

Generic, 'average' story: standard structure (setup,
discovery, ending), common words and the most
expected plot, for example the message comes from
a lost sailor.

2. Style-guided
Write a short story about a lighthouse keeper who
finds a message in a bottle, as a gentle bedtime fable
for children with warm simple language, a talking
seagull and a soft moral at the end.

Distinct voice: shorter sentences, repetition, friendly
tone, a character not in the neutral version, and an
explicit moral.

3.
Constraint-based

Write exactly five sentences about a lighthouse keeper
who finds a message in a bottle. Include the words
'storm' and 'compass', use no dialogue, and end with a
question.

Very compact and structured: fixed length, required
words present, final line is a question. Less freedom,
so less variation between runs, and easier to check
automatically.

Why high temperature is more creative but less consistent

• More creative: a flatter distribution lets rare words and unusual combinations appear (for example 'the lamp
hummed a secret'), giving surprise and fresher imagery.

• Less consistent: the same flattening also lets bad choices through (off-topic words, grammar slips,
contradictions, a changed character name).

• Errors compound: every sampled token becomes part of the context for all later tokens, so one odd choice
can push the story in a different direction.

• Not repeatable: two runs with the same prompt can be very different, which is bad for factual or brand
writing.

• Rule of thumb: T about 0.2 for facts and code, about 0.7 for balanced writing, 1.0 or more for poetry and
brainstorming (usually paired with top-p to cut the worst tokens).

Question 3

Explain how AI music generation works at a high level: representing music as symbols (MIDI) or as audio, sequence
models, and diffusion or transformer models for audio. List three challenges of generating music (long-term
structure, quality, rights over training data) and one way to handle each.

Answer:

1. How music is represented

• Symbolic (MIDI): music is a list of events: which note (pitch 0 to 127), when it starts, how long it lasts, how
hard it is played (velocity) and which instrument. It is turned into tokens such as NOTE_ON_60,
TIME_SHIFT_250ms, NOTE_OFF_60. It is compact (a song is a few thousand tokens) and editable, but it
has no real sound or voice.

• Audio: a waveform (44,100 numbers per second, so a 3-minute song is about 8 million numbers) or a
spectrogram (a picture of frequency against time). It captures timbre, vocals and production but is huge.
Modern systems first compress audio with a neural audio codec (such as EnCodec or SoundStream) into a
few discrete tokens per second.

2. Models used

• Sequence models: RNN/LSTM (early work), and transformers (Music Transformer, MuseNet, MusicGen)
predict the next note or audio token exactly like a language model predicts the next word. A decoder (codec
or vocoder) converts the tokens to sound.

• Diffusion models: start from noise and denoise step by step into a spectrogram or a compressed latent
audio, guided by a text embedding (Riffusion, AudioLDM, Stable Audio). They give high quality and good

text control.

• Typical pipeline: text prompt -> text encoder -> transformer over codec tokens (or latent diffusion) ->
decoder / vocoder -> audio file.

3. Challenges and one way to handle each

Challenge
Why it is hard
One way to handle it

Long-term structure
Songs have verse, chorus, bridge, repeated themes,
a fixed key and tempo over minutes. Models have
limited context and tend to wander or lose the
theme.

Generate hierarchically: first a plan (chords,
sections), then the details; use long-context
transformers or relative attention; condition on
section labels.

Quality
Audio has very fine detail; artifacts, noise, off-key or
unnatural vocals are easy to hear, and quality is hard
to score.

Use high-fidelity neural codecs plus diffusion
up-sampling, and check with human listening tests
and metrics such as Frechet Audio Distance.

Rights over training
data

Models may be trained on copyrighted recordings
without permission, and can imitate a living artist's
voice or style.

Train on licensed or royalty-free music, offer artist
opt-out, share revenue, and watermark and label
AI output.

Question 4

Write a Python program that uses a very simple Markov chain (a dictionary of word to next-word lists) built from a
short paragraph of your choice to generate a 20-word piece of text, with a fixed random seed. Explain how this toy
model is different from a large language model and why its output is less coherent.

Answer:

Python program

import random

In [ ]:
import random
# Short paragraph used to build the model
text = ("The sun rose over the quiet village and the children ran to the river. "
        "The river was cold and the water was clear. The children laughed and "
        "the village woke up. The sun was warm and the day was long.")
# Step 1: build a dictionary of word -> list of next words
words = text.lower().replace(".", "").split()
chain = {}
for current, nxt in zip(words, words[1:]):
    chain.setdefault(current, []).append(nxt)
# Step 2: generate 20 words with a fixed seed
random.seed(42)
word = random.choice(words)
output = [word]
while len(output) < 20:
    options = chain.get(word)
    if not options:                 # dead end: restart from a random word
        options = words
    word = random.choice(options)
    output.append(word)
print("Chain for 'the':", chain["the"])
print("Generated text :", " ".join(output))

Output (identical on every run because of random.seed(42))

Chain for 'the': ['sun', 'quiet', 'children', 'river', 'river', 'water', 'children',
                 'village', 'sun', 'day']
Generated text : and the river the river the quiet village woke up the quiet village and the
                 sun rose over the village

(Long output lines are wrapped here only to fit the page.)

How it works

The dictionary maps each word to every word that followed it in the paragraph. For 'the' the list has 10 entries,
and 'river' appears twice, so it is twice as likely to be picked. Generation repeatedly looks at the current word,
picks a random follower, and moves on. This is next-word prediction in its simplest form.

Difference from a large language model (LLM)

Feature
Toy Markov chain
Large language model

Context used
Only the last 1 word
Thousands or more previous tokens (attention)

Training data
One short paragraph (about 40 words)
Trillions of tokens from books, web, code

Learning
Just counts which word follows which
Billions of learned parameters (transformer neural
network)

Meaning
No idea of meaning, grammar or topic
Embeddings capture meaning, syntax and facts

Unseen input
Fails or restarts if a word was never seen
Generalises to new words and sentences

Instructions
Cannot follow a prompt
Can follow instructions, style and constraints

Why its output is less coherent

• It has a memory of one word, so it cannot remember the subject of the sentence. In the output, 'the river the
river the quiet village ... the quiet village' loops and jumps topic.

• It can only copy word pairs it has seen, so it has no real grammar rules and no plan for where the sentence
is going.

• The training text is tiny, so there are few choices per word and loops are very likely.

• It has no idea of meaning or of the overall topic, so nothing makes sentences connect. An LLM models
long-range dependencies and meaning, which is why its text stays on topic.

Question 5

Explain the ethical issues of creative generative AI: copyright and training data, originality and credit to artists,
plagiarism, deepfakes and misinformation, and the effect on creative jobs. Give one real or realistic example for
each issue and one possible response (for example licensing, watermarking, disclosure, opt-out).

Answer:

Issue
What is the problem?
Example
One possible response

Copyright and
training data

Models are trained on huge
amounts of scraped text, images
and music, often without
permission or payment to the
creators.

Lawsuits such as Getty Images v.
Stability AI (images allegedly used to
train an image model) and The New
York Times v. OpenAI (articles
allegedly used for training).

Licensing deals, training on
licensed or public-domain data,
opt-out tools for creators, and
transparency about datasets.

Originality and
credit to artists

Models can imitate a living artist's
style on request; the artist gets no
credit or income, and it is unclear
who owns the result.

A user prompts 'in the style of
[named illustrator]' and sells the
image; in 2022 an AI-assisted image
won a state-fair art contest, which
caused debate about disclosure and
credit.

Credit and disclosure rules, style
opt-out lists, competition rules
requiring labels, revenue sharing.

Plagiarism
A model can reproduce
memorised passages, and people
can submit AI-written work as
their own.

A student submits an AI-written
essay; a model repeats a paragraph
from a published article almost word
for word.

Disclosure policies, similarity and
plagiarism checks, citing AI use,
assessments such as oral
defence or in-class work.

Issue
What is the problem?
Example
One possible response

Deepfakes and
misinformation

Realistic fake voices, images and
videos can deceive people,
damage reputations and
influence elections.

A fake robocall that copied a political
leader's voice in the 2024 US
primaries; fake celebrity videos used
in investment scams.

Watermarking and content
credentials (C2PA, SynthID),
platform labels and takedown,
detection tools, laws and media
literacy.

Effect on
creative jobs

Cheap, fast AI output can reduce
paid work for illustrators, writers,
voice actors and musicians,
especially at entry level.

Freelance illustrators and copywriters
reporting fewer commissions; the
2023 Hollywood writers and actors
strikes included demands for AI
protections.

Consent and fair pay rules,
contracts that limit AI use,
retraining, and keeping humans
in creative decision roles.

Balanced view: generative AI also lowers the barrier to creating (accessibility, small creators, faster
prototyping). The aim of the responses above is not to ban it but to make sure creators are consulted, credited
and paid, and that audiences are not deceived.

Question 6

Explain responsible-use practices for generative AI content: disclosure that content is AI-generated, watermarking
and content provenance, human review, bias checks and safe-content filters. Write a short policy of at least eight
rules for a company that uses generative AI to write marketing content.

Answer:

Responsible-use practices

• Disclosure: tell people when content is AI-generated or AI-assisted, so they can judge it properly and are
not misled. Use clear labels such as 'Created with AI'.

• Watermarking and provenance: a watermark is a hidden or visible signal that marks content as AI-made
(for example SynthID). Provenance means a verifiable record of how a file was made and edited (for
example C2PA content credentials). Both help detect fakes and keep a chain of trust.

• Human review: a person checks facts, tone, legal risk and brand fit before publishing, because models can
invent facts (hallucinate) and have no accountability.

• Bias checks: test outputs for stereotypes and unequal treatment of groups (gender, region, religion, age),
with diverse reviewers and sample prompts.

• Safe-content filters: automatic checks on inputs and outputs that block hateful, violent, sexual, deceptive
or otherwise harmful content before it reaches the audience.

Company policy: use of generative AI for marketing content

1. Disclose. Label AI-generated images, video, voice and chatbots clearly, and follow advertising and
platform disclosure rules.

2. Human sign-off. No AI-written content is published until a named human editor has reviewed and
approved it.

3. Fact-check every claim. Prices, statistics, product features and comparisons must be verified against
approved sources. Never publish AI-invented quotes, reviews or testimonials.

4. Approved tools only. Staff may use only company-approved AI tools and accounts.

5. Protect data. Never enter customer personal data, confidential plans or unreleased product details into
external AI tools.

6. Respect intellectual property. Do not prompt for the style or work of a named living artist or writer, or for
other brands' logos or characters; use licensed assets and check output for similarity to existing work.

7. Check for bias. Review content for stereotypes and exclusion; test important campaigns with diverse
reviewers.

8. Keep it safe. Use content filters; never produce deceptive, discriminatory, sexual or violent content, and
never target children or vulnerable groups with manipulative messages.

9. No deepfakes without consent. Do not create the face, voice or likeness of a real person, or fake
endorsements, without written consent.

10. Keep records and audit. Log the prompt, model and version, date and reviewer for each piece; use
watermarks or content credentials where tools support them; report errors and remove faulty content within
24 hours; review this policy every quarter.

Question 7

Explain hyper-personalization with generative AI (for example personalized stories, learning paths, product
recommendations and marketing messages). Describe the data used, the benefits, and the risks (privacy,
manipulation, filter bubbles, bias). Design a personalized learning assistant: what user data you would use, what you
would not collect, and how the user can control it.

Answer:

What is hyper-personalization?

Hyper-personalization means adapting content to each individual in real time using their data. Older
personalization picked from fixed templates ('customers who bought X also bought Y'). With generative AI the
content itself is created for the person: the wording, examples, difficulty, image or offer are generated for that
one user.

Use case
How generative AI personalizes it

Personalized stories
A child's name, interests, reading level and language are put into the story, so the hero is like the
child.

Learning paths
Explanations, examples and practice questions are generated at the learner's level, focused on the
topics they missed.

Product
recommendations

Suggestions plus generated descriptions that explain why an item fits this shopper.

Marketing messages
Email subject, tone, offer and timing written for each customer's behaviour and preferences.

Data used

• Profile: age group, language, location (broad), stated interests and goals.

• Behaviour: clicks, purchases, time spent, quiz results, topics skipped or repeated.

• Context: device, time of day, current session activity.

• Feedback: ratings, likes and dislikes, 'too hard' or 'too easy' flags.

Benefits

• Higher relevance and engagement; people see what helps them.

• Better learning results because difficulty and examples match the learner.

• Accessibility: content in the user's language, reading level or format.

• Efficiency and better conversion for businesses; less irrelevant content for users.

Risks and mitigations

Risk
Explanation
Mitigation

Privacy
Detailed profiles can reveal sensitive facts (health, finances,
beliefs) and can be leaked or misused.

Collect the minimum, encrypt, set retention
limits, ask for consent.

Manipulation
Messages can exploit emotions or weaknesses (for example
urgency pressure on a vulnerable user).

Ban dark patterns, set ethics rules, review
campaigns, offer easy opt-out.

Filter bubbles
Showing only what a user already likes narrows their view
and stops them discovering new ideas.

Mix in diverse and new items; let users reset
recommendations.

Bias
Models may stereotype by gender, region or income, so
groups get different quality of service.

Audit outputs across groups; avoid sensitive
attributes as inputs.

Design: personalized learning assistant ('StudyBuddy')

Data I WOULD use (and why)
Data I would NOT collect
How the user controls it

Chosen subject and goal (to set the path)
Self-declared level or grade (to start at
the right difficulty)
Quiz scores and topics missed (to focus
practice)
Time spent per topic (to pace lessons)
Preferred explanation style: examples,
step-by-step or visual
Language and study schedule (optional)

Real name (a nickname is enough)
Exact location, contacts or photos
Microphone or camera data unless the
user starts it
Health, religion, family or financial
details
Activity outside the app
Data from children without parental
consent
No selling or sharing of data with
advertisers

'My data' page to view everything stored
Edit or delete any item, or all data, at any
time
Export data
Switch personalization off (generic mode)
Pause history
Opt in (off by default) to use data to
improve the model
'Why am I seeing this?' explanations
Auto-delete after 12 months of inactivity;
parental controls for under 18

How it works: quiz results update a small learner profile. The profile summary (not raw history) is placed in the
prompt of the language model, which writes an explanation and practice questions at that level. The learner's
rating feeds back into the profile. To avoid a filter bubble, the assistant regularly adds topics slightly outside the
learner's comfort zone.

Question 8

Explain how reinforcement learning is used with generative models: reward signals, reinforcement learning from
human feedback (RLHF), and reward design for creative tasks (how do you reward a good poem?). Explain reward
hacking with an example and why creative quality is hard to measure automatically.

Answer:

Why reinforcement learning (RL) is used

Pretraining only teaches a model to imitate text (predict the next token). It does not teach what people actually
find helpful, safe or well written. RL fine-tunes the model to maximise a reward, a number that says how good
an output is.

Reward signals

• Human preference: people compare outputs and say which is better.

• Automatic rules: format correct, length limit, unit tests passed, banned words absent.

• Model-based: a trained reward model (or another AI) scores outputs.

RLHF: reinforcement learning from human feedback

• Step 1 - Pretrain the language model on a large text corpus.

• Step 2 - Supervised fine-tuning (SFT) on examples of good responses written by people.

• Step 3 - Collect preferences: the model writes several answers to the same prompt and human raters rank
them.

• Step 4 - Train a reward model to predict those rankings, so it gives a higher score to the answers humans
prefer.

• Step 5 - Optimise the policy (the language model) with an RL algorithm such as PPO to get high reward,
while a KL penalty keeps it close to the original model so it does not drift into strange text. Simpler
alternatives such as DPO use the preference pairs directly.

maximize   E[ reward(prompt, answer) ]  -  beta * KL( new_model || reference_model )

Reward design for a creative task: how do you reward a good poem?

There is no single correct poem, so one number is built from several parts. An illustrative design:

reward = 0.50 * human_preference      # raters score imagery, emotion, originality (1-5)
       + 0.20 * form_match            # asked for a haiku? 5-7-5 syllables; sonnet? rhyme scheme
       + 0.15 * relevance             # stays on the requested theme
       + 0.15 * novelty               # not too close to existing poems or to its own earlier outputs
       - penalty_if_copied_or_unsafe  # large negative for plagiarism or harmful content

Reward hacking

Reward hacking happens when the model finds a way to get a high reward without doing what we really wanted
(Goodhart's law: when a measure becomes a target, it stops being a good measure).

Example: suppose we automatically reward poems by the number of rhyming word pairs. The model learns to
write: 'The cat sat on a hat, a bat, a mat, a gnat.' The rhyme count is maximal, but the poem is meaningless.
Another common case: if raters tend to prefer long, confident answers, the reward model learns this and the
model becomes wordy and over-flattering.

Reducing it: combine several rewards, keep the KL penalty, audit samples by humans, refresh the reward
model with new ratings, and test with adversarial prompts.

Why creative quality is hard to measure automatically

• Subjective: taste differs between people, cultures and moments (a funeral poem and a birthday poem are
judged differently).

• No ground truth: there is no single 'correct' poem to compare with, unlike a maths answer.

• Many dimensions: originality, emotion, rhythm, imagery and coherence all matter and can conflict.

• Novelty paradox: too familiar is boring, too different looks like an error.

• Bad proxies: metrics such as BLEU or perplexity measure similarity to reference text, not creativity.

• Rater bias and disagreement: raters prefer longer or familiar styles and often disagree with each other.

Question 9

Explain multimodal systems: models that handle text, images, audio and video together. Explain the idea of a shared
embedding space and cross-modal learning (for example text-to-image, image captioning, speech-to-text and
text-to-speech). Draw a block diagram of a system that takes an image and a question and produces a spoken
answer.

Answer:

Multimodal systems

A modality is a type of data: text, image, audio or video. A multimodal system can take in, relate and sometimes
produce more than one modality, for example describing a photo in words, answering a spoken question about
a picture, or turning a script into a video.

Shared embedding space

An embedding is a list of numbers (a vector) that represents meaning. In a shared embedding space, an image
encoder and a text encoder map their inputs into the same vector space so that things with the same meaning
are close together, whatever the modality. The photo of a dog and the words 'a dog' end up near each other;
the words 'a plane' end up far away.

Example (CLIP): trained on hundreds of millions of image-caption pairs with contrastive learning: matching
image and caption vectors are pulled together (high cosine similarity), non-matching pairs are pushed apart.
This enables zero-shot image classification, image search by text, and guiding text-to-image models.

Cross-modal learning

Cross-modal learning means using what is learned in one modality to understand or generate another.

Task
Input -> Output
Example model
Idea

Text-to-image
Text -> Image
Stable Diffusion, DALL-E
Text embedding guides the denoising of
an image.

Image captioning
Image -> Text
BLIP, vision-language models
Vision encoder features are fed to a
language decoder that writes the caption.

Speech-to-text (ASR)
Audio -> Text
Whisper
Audio features are decoded into words.

Text-to-speech (TTS)
Text -> Audio
Neural TTS such as VITS
Text is converted to a spectrogram and
then to a waveform.

Visual question
answering

Image + Text -> Text
Multimodal LLMs
Image tokens and question tokens are
processed together.

Block diagram: image + question -> spoken answer

Input 1: Image
Input 2: Question
(spoken or typed)

Vision encoder
(for example ViT / CLIP image encoder)

Speech-to-text (only if spoken)

+ tokenizer

Image embeddings

Text embeddings

(visual tokens)

(text tokens)

Multimodal language model
fuses image and question in one shared embedding space
(cross-attention or projected visual tokens) and writes the answer

Text answer
Text-to-speech

Output: spoken answer

(neural TTS)

(audio)

Flow: the image goes through a vision encoder and the question (converted to text if spoken) through a
tokenizer; both become embeddings; the multimodal language model combines them and writes a text answer;
a text-to-speech model turns that answer into audio.

Question 10

Mini project: Design a small multimodal creative application (for example a children's story app that writes a story
from a prompt, generates pictures for each page and reads the story aloud). Write a one-page design that lists the
models or APIs you would use for each step, the prompts, the data flow, the ethical safeguards (disclosure,
safe-content filters, privacy), how you would use human feedback to improve it, and how you would evaluate the
quality. Optionally implement the text-generation step in Python with any LLM API (use a placeholder API key).

Answer:

Project: 'StoryBook Studio' - a children's story app

A parent or child types an idea (for example 'a shy dragon who learns to share'), chooses an age group and
language. The app writes a 6-page story, draws a picture for every page and reads each page aloud. A parent
can preview the story before the child reads it.

1. Models / APIs, prompts and outputs

Step
Model / API (examples)
Prompt or input
Output

1. Input check
Moderation API (for example
OpenAI Moderation) or a small
classifier

The user's idea text
Allowed or blocked, with a
friendly message if blocked

2. Write story
LLM API (Claude, GPT, Gemini),
temperature about 0.8

System: 'You are a children's picture-book
author. Gentle, age-appropriate, simple
words, kind message, no violence, no
brands, original characters. Reply in JSON.'
User: idea + age group + 6 pages

JSON: title, character sheet, 6
pages each with text and
image_description

3. Story check
Moderation API + readability
score (Flesch-Kincaid) + rule
checks

Generated story text
Pass, or regenerate
(maximum 3 tries), or send to
human review

4. Illustrations
Text-to-image model (Stable
Diffusion XL, DALL-E 3, Imagen)

'Children's picture-book illustration, soft
watercolour, [image_description]. Character:
[character sheet]. Same character and
colours as the other pages. No text in the
image.'

One image per page (6
images)

5. Image
check

Image safety classifier
Each generated image
Pass, or regenerate

6. Narration
Text-to-speech API (Google
Cloud TTS, ElevenLabs, OpenAI
TTS)

Page text, warm friendly voice, speed 0.9
One audio file per page

7. Assemble
App backend
Text + image + audio per page
Story book screen with 'Made
with AI' label

2. Data flow

Parent / child idea
        |
        v
 [1 Input moderation] --blocked--> friendly "let's try another idea" message
        |
        v
 [2 Story LLM] -> JSON: title, character sheet, 6 x (text + image_description)

|
        v
 [3 Story safety + reading-level check] --fail--> regenerate (max 3) / human review
        |
        +----------------------------+
        v                            v
 [4 Text-to-image x 6]        [6 Text-to-speech x 6]
        |                            |
        v                            |
 [5 Image safety check]              |
        |                            |
        +-------------+--------------+
                      v
 [7 Assemble pages: text + picture + audio, add "Made with AI" label]
                      |
                      v
 [Parent preview -> child reads -> thumbs / flags -> feedback store]

3. Ethical safeguards

• Disclosure: 'Made with AI' label on the cover and in the app description; parents are told which kinds of
models are used.

• Safe-content filters: checks on the input, the story text and every image; kid-safe rules (nothing scary,
violent or romantic, no brands, no real people); a report button for anything that slips through.

• Privacy: children's data is minimal. Parental consent (COPPA / GDPR-K style rules); no real names or
photos sent to AI APIs; prompts auto-deleted after 30 days; no use of children's data for training; no ads.

• Originality and IP: prompts require original characters; no 'in the style of' a named living artist; no
copyrighted characters.

• Bias: test that characters, families and settings are diverse and not stereotyped.

• Human oversight: parent preview, manual review of flagged stories, screen-time reminders.

4. Using human feedback to improve it

• Thumbs up/down for each story and each picture, plus quick tags: 'too scary', 'too hard', 'boring', 'picture
does not match'.

• Parents and teachers rate a weekly sample of stories on a 1 to 5 scale; every rating is stored with the
prompt and the output.

• Use the ratings to improve the system prompt, to pick good examples to include in the prompt, and (with
consent and enough data) to fine-tune or preference-tune the model (RLHF/DPO) on better-versus-worse
pairs.

• A/B test prompt versions and keep the one with higher ratings and fewer flags.

5. Evaluating quality

Dimension
Metric
How it is measured

Story quality
Average human rating (1-5): coherence, creativity, kindness
of message

Parent and teacher panels

Readability
Flesch-Kincaid grade close to the age target
Automatic score on each story

Safety
Share of outputs blocked or flagged; zero harmful items
Red-team test of 200 tricky prompts
before each release

Picture-text match
CLIP similarity between page text and image; character
consistency

Automatic score plus human spot
check

Narration
Word error rate when the audio is transcribed; listening
score

Speech-to-text check, listener ratings

Engagement and cost
Story completion rate, re-reads, seconds and cost per story
App analytics

6. Python: text-generation step (placeholder API key)

The code runs even without a real key (it returns a small demo story). With a real key it calls the LLM API.

import json

In [ ]:
import json
try:
    from anthropic import Anthropic          # pip install anthropic
except ImportError:
    Anthropic = None
API_KEY = "YOUR_API_KEY_HERE"                # placeholder - never hard-code a real key
SYSTEM_PROMPT = (
    "You are a children's picture-book author. Write warm, gentle, age-appropriate "
    "stories with simple words and a kind message. No violence, no scary scenes, "
    "no brand names, no real people. Always invent original characters. "
    "Reply ONLY with valid JSON."
)
BLOCKED_WORDS = {"gun", "kill", "blood", "weapon"}   # toy input filter
def is_prompt_safe(idea):
    return not any(w in idea.lower() for w in BLOCKED_WORDS)
def generate_story(idea, age_group="5-7", pages=6):
    if not is_prompt_safe(idea):
        raise ValueError("Sorry, let's pick a different, friendlier idea!")
    user_prompt = (
        f"Story idea: {idea}\nAge group: {age_group}\n"
        f"Write exactly {pages} pages. Return JSON like: "
        '{"title": "...", "character_sheet": "...", '
        '"pages": [{"text": "...", "image_description": "..."}]}'
    )
    # Offline demo path so the code runs without a real key
    if Anthropic is None or API_KEY.startswith("YOUR_"):
        return {
            "title": "Dara the Dragon Learns to Share",
            "character_sheet": "Dara: small green dragon, round yellow eyes, red scarf",
            "pages": [{"text": "Dara had a huge pile of shiny berries.",
                       "image_description": "Dara guarding a pile of berries"}] * pages,
        }
    client = Anthropic(api_key=API_KEY)
    response = client.messages.create(
        model="claude-sonnet-5-5",           # any LLM works here
        max_tokens=1500,
        temperature=0.8,                     # some creativity, still consistent
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return json.loads(response.content[0].text)
if __name__ == "__main__":
    story = generate_story("a shy dragon who learns to share")
    print("Title    :", story["title"])
    print("Character:", story["character_sheet"])
    print("Pages    :", len(story["pages"]))
    print("Page 1   :", story["pages"][0]["text"])

Output (demo mode):

Title    : Dara the Dragon Learns to Share
Character: Dara: small green dragon, round yellow eyes, red scarf

Pages    : 6
Page 1   : Dara had a huge pile of shiny berries.